Dataset link: https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks

The sonar dataset (also known as the “mines vs rocks” dataset) contains the patterns generated by bouncing sonar signals off two different types of objects: metal cylinders (M), (which could potentially be mines), and rocks (R).

We will implement multi-class classifier from scratch to predict Mine or Rock. As these are only 2 class we can use binary classification too.

Activation Function used - Sigmoid
Loss Function - Log Loss
Gradient - Partial derivation of loss       => Grap has (weights to loss)
Weights - dimension of weights is number of features (60 + 1 for bias) , number of classes (2) => (61,2)

Input dimension - (208,61) => 208 examples, 60 features and 1 label column

In [1]:
import csv
import numpy as np

In [6]:
data_as_python_list = list(csv.reader(open("Datasets/connectionist+bench+sonar+mines+vs+rocks/sonar.all-data")))

data_without_bias_column = np.array(data_as_python_list)

#Prepend bias column, resulting in a (208,62) matrix:

data = np.insert(data_without_bias_column,0,1,axis=1)


In [7]:
# Shuffle data. This is important, because the Sonar dataset contains all
# "Rock" examples first, and all "Metal" examples later. If we don't shuffle it,
# we'll end up with a test set composed exclusively of "Metal" examples when
# we split the dataset later.

np.random.seed(1234)
np.random.shuffle(data)

In [9]:
#Seperate features and labels into X and Y

X = data[:,0:-1].astype(np.float32)
labels = data[:,-1].reshape(-1,1)

Y_unencoded = (labels == 'M').astype(np.int64)

#split data now into training and testing

SIZE_OF_TRAINING_SET = 160

X_train,X_test = np.vsplit(X,[SIZE_OF_TRAINING_SET])
Y_train_unencoded,Y_test = np.vsplit(Y_unencoded,[SIZE_OF_TRAINING_SET])

In [14]:
#One hot encode the training set, but not the test set:

def One_hot_encode(Y):
    n_labels = Y.shape[0]
    n_classes = 2
    encoded_Y = np.zeros((n_labels,n_classes))

    for i in range(n_labels):
        label = Y[i]
        encoded_Y[i][label] = 1
    return encoded_Y

Y_train = One_hot_encode(Y_train_unencoded)

In [15]:
#The Real thing start here

#Applying Logistic Regression
def sigmoid(z):
    return 1/ (1+np.exp(-z))

#performing forward propogation
def forward(X,w):
    weighted_sum = np.matmul(X,w)
    return sigmoid(weighted_sum)

#Calling predict function

def classify(X,w):
    y_hat = forward(X,w)
    labels = np.argmax(y_hat,axis=1)
    return labels.reshape(-1,1)

#computing Loss over using logistic regression
def loss(X,Y,w):
    y_hat = forward(X,w)
    first_term = Y*np.log(y_hat)
    second_term = (1-Y) * np.log(1-y_hat)
    return -np.sum(first_term+second_term) / X.shape[0]

#Calculating gradient
def gradient(X,Y,w):
    return np.matmul(X.T,(forward(X,w) - Y)) / X.shape[0]

In [16]:
#Printing results
def report(iteration,X_train,Y_train,X_test,Y_test,w):
    matches = np.count_nonzero(classify(X_test,w) == Y_test)
    n_test_examples = Y_test.shape[0]
    matches = matches * 100.0 / n_test_examples
    training_loss = loss(X_train,Y_train,w)
    print("%d - Loss: %.20f, %2f%%" % (iteration, training_loss,matches))


#Calling the training function for desired no. of iterations
def train(X_train,Y_train,X_test,Y_test,iterations,lr):
    w = np.zeros((X_train.shape[1], Y_train.shape[1]))
    for i in range(iterations):
        report(i,X_train,Y_train,X_test,Y_test,w)
        w-= gradient(X_train,Y_train,w) * lr
    report(iterations,X_train,Y_train,X_test,Y_test,w)
    return w

In [17]:
w = train(X_train,Y_train,X_test,Y_test,10000,0.01)

0 - Loss: 1.38629436111989057245, 45.833333%
1 - Loss: 1.38574237646475784125, 54.166667%
2 - Loss: 1.38520299652202782958, 54.166667%
3 - Loss: 1.38467568598360690757, 54.166667%
4 - Loss: 1.38415993276702886661, 54.166667%
5 - Loss: 1.38365524701695941090, 54.166667%
6 - Loss: 1.38316116014845524873, 54.166667%
7 - Loss: 1.38267722393035619177, 54.166667%
8 - Loss: 1.38220300960723463390, 54.166667%
9 - Loss: 1.38173810705837452062, 54.166667%
10 - Loss: 1.38128212399231253826, 54.166667%
11 - Loss: 1.38083468517551266608, 54.166667%
12 - Loss: 1.38039543169380252152, 54.166667%
13 - Loss: 1.37996402024524367214, 54.166667%
14 - Loss: 1.37954012246315715906, 54.166667%
15 - Loss: 1.37912342426807099649, 54.166667%
16 - Loss: 1.37871362524740126432, 54.166667%
17 - Loss: 1.37831043806172237609, 54.166667%
18 - Loss: 1.37791358787652784557, 54.166667%
19 - Loss: 1.37752281181841662594, 54.166667%
20 - Loss: 1.37713785845469294244, 54.166667%
21 - Loss: 1.37675848729539440640, 54.166667